In [ ]:
import numpy as np
from sklearn.metrics import silhouette_score
from scipy.spatial.distance import euclidean

# ============================================================================
# STEP 1: Define the three clusters with their data points
# ============================================================================
clusters = [
    np.array([[2,5], [3,4], [4,6]]),      # Cluster 1: 3 points
    np.array([[8,3], [9,2], [10,5]]),     # Cluster 2: 3 points
    np.array([[6,10], [7,8], [8,9]])      # Cluster 3: 3 points
]

# ============================================================================
# STEP 2: Prepare data for silhouette score calculation
# ============================================================================
# Combine all points from all clusters into one dataset
X = np.vstack(clusters)

# Create labels: assign each point to its cluster (0, 1, or 2)
# Example: [0,0,0, 1,1,1, 2,2,2] means first 3 points belong to cluster 0, etc.
labels = np.repeat(np.arange(len(clusters)), len(clusters[0]))

# ============================================================================
# STEP 3: Calculate overall silhouette score
# ============================================================================
# The silhouette score ranges from -1 to 1
# Higher values indicate better-separated clusters
silhouette_avg = silhouette_score(X, labels)
print(f"Average silhouette score: {silhouette_avg:.4f}")

# ============================================================================
# STEP 4: Define function to calculate silhouette coefficient for a single point
# ============================================================================
def silhouette_for_point(point, same_cluster, other_clusters):
    """
    Calculate silhouette coefficient for one data point.
    
    Parameters:
    -----------
    point : list
        The data point to evaluate (e.g., [2, 5])
    same_cluster : numpy array
        All points in the same cluster as the target point
    other_clusters : list of numpy arrays
        All other clusters' points
    
    Returns:
    --------
    s : float
        Silhouette coefficient (-1 to 1)
    a : float
        Average distance to points in same cluster
    b : float
        Minimum average distance to other clusters
    """
    
    # Calculate 'a': average distance to all points in same cluster (excluding itself)
    distances_to_same_cluster = []
    for p in same_cluster:
        if not np.array_equal(point, p):  # Exclude the point itself
            distances_to_same_cluster.append(euclidean(point, p))
    
    # If no distances found, set 'a' to 0
    if len(distances_to_same_cluster) == 0:
        a = 0
    else:
        a = np.mean(distances_to_same_cluster)
    
    # Calculate 'b': minimum average distance to any other cluster
    min_distance_to_other = float('inf')
    for other_cluster in other_clusters:
        # Calculate average distance to this other cluster
        avg_distance = np.mean([euclidean(point, p) for p in other_cluster])
        # Keep track of the minimum
        if avg_distance < min_distance_to_other:
            min_distance_to_other = avg_distance
    b = min_distance_to_other
    
    # Calculate silhouette coefficient: s = (b - a) / max(a, b)
    s = (b - a) / max(a, b)
    
    return s, a, b

# ============================================================================
# STEP 5: Calculate silhouette for specific points
# ============================================================================
print("\n" + "="*50)
print("Manual Silhouette Calculation Results")
print("="*50)

# Test point 1: A1 (2,5) from Cluster 1
point_A1 = [2, 5]
cluster_idx_A1 = 0
other_clusters_A1 = [clusters[1], clusters[2]]
s_A1, a_A1, b_A1 = silhouette_for_point(point_A1, clusters[cluster_idx_A1], other_clusters_A1)
print(f"Point A1 (2,5): a={a_A1:.3f}, b={b_A1:.3f}, s={s_A1:.3f}")

# Test point 2: B2 (9,2) from Cluster 2
point_B2 = [9, 2]
cluster_idx_B2 = 1
other_clusters_B2 = [clusters[0], clusters[2]]
s_B2, a_B2, b_B2 = silhouette_for_point(point_B2, clusters[cluster_idx_B2], other_clusters_B2)
print(f"Point B2 (9,2): a={a_B2:.3f}, b={b_B2:.3f}, s={s_B2:.3f}")

Average silhouette score: 0.6149

=== Manual Calculation Results ===
A1 (2,5): a=1.825, b=6.482, s=0.718
B2 (9,2): a=2.288, b=6.781, s=0.663
